# M2.Ex2: Automobile Fuel Efficiency

- Run: [**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/B5/blob/main/content/W3/M2/exercises/ex2_multi-reg.ipynb)

## Auto MPG Dataset

- Features: `5` numerical, `3` categorical
- Target: `mpg` (miles per gallon)
- Size: `398` samples
- Source: [Auto MPG Dataset](https://archive.ics.uci.edu/dataset/9/auto+mpg)

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

### Step 1. Load the data

In [ ]:
# Load from UCI via pandas (whitespace-separated, '?' for missing values)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
col_names = ['mpg', 'cylinders', 'displacement', 'horsepower',
             'weight', 'acceleration', 'model year', 'origin', 'car name']

df = pd.read_csv(url, sep='\s+', names=col_names, na_values='?')
display(df.head())
print(f'Shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()}')

### Step 2.a Assign variables `X` to the features and `y` to the target

In [ ]:
# Drop 'car name' — it's a unique identifier, not a predictive feature
# Keep all other columns as features; target is 'mpg'
X = df.drop(columns=['mpg', 'car name'])
y = df['mpg']

print('Features:', X.columns.tolist())
print('X shape :', X.shape)
print('y shape :', y.shape)
display(X.head())

### Step 2.b print the type of each

In [ ]:
print('Type of X:', type(X))
print('Type of y:', type(y))
print()
print('Column dtypes:')
print(X.dtypes)

### Step 2.c identify whether the target is categorical or numerical & whether the task is regression or classification

In [ ]:
print('Target dtype:', y.dtype)
print()
print('The target (mpg) is NUMERICAL — it is a continuous float value.')
print('Therefore, the task is REGRESSION.')

### Step 3. Identify the number of samples and columns of both the data matrix and the target

In [ ]:
print('--- Feature Matrix X ---')
print(f'Shape            : {X.shape}')
print(f'Number of samples: {X.shape[0]}')
print(f'Number of columns: {X.shape[1]}')
print()
print('--- Target Vector y ---')
print(f'Shape            : {y.shape}')
print(f'Number of samples: {y.shape[0]}')

### Step 4. Summarize the distribution of the data

- Use `describe()` for numerical features
- Use `describe()` for categorical features

In [ ]:
# Identify numerical vs categorical-like columns
# 'cylinders', 'model year', 'origin' are discrete/categorical in nature
numerical_cols    = ['displacement', 'horsepower', 'weight', 'acceleration']
categorical_cols  = ['cylinders', 'model year', 'origin']

print('=== Numerical Features ===')
display(df[numerical_cols + ['mpg']].describe())

print('\n=== Categorical-like Features ===')
display(df[categorical_cols].describe())

print('\nValue counts for origin (1=USA, 2=Europe, 3=Japan):')
print(df['origin'].value_counts().sort_index())
print('\nValue counts for cylinders:')
print(df['cylinders'].value_counts().sort_index())

### Step 5. Plot each of the features vs the target

In [ ]:
# Pairplot: numerical features vs mpg
num_features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']

sns.pairplot(df[num_features + ['mpg']],
             x_vars=num_features,
             y_vars='mpg',
             height=4, aspect=0.75,
             kind='reg',
             plot_kws={'line_kws': {'color': 'crimson'}, 'scatter_kws': {'alpha': 0.3}})
plt.suptitle('Auto Features vs MPG', y=1.02, fontsize=13, fontweight='bold')
plt.show()

### Step 6. What is the relationship between the feature and the target? (increasing or decreasing or none)

1. `x=cylinders` and `y=mpg`
2. `x=displacement` and `y=mpg`
3. `x=horsepower` and `y=mpg`
4. `x=weight` and `y=mpg`
5. `x=acceleration` and `y=mpg`

In [ ]:
num_df = df[num_features + ['mpg']].dropna()
corr = num_df.corr()['mpg'].drop('mpg')
print('Pearson Correlation with mpg:')
print(corr.round(4))
print()
print('1. cylinders    vs mpg: DECREASING (strong negative, r ≈ -0.78)')
print('   More cylinders → heavier engine → lower fuel efficiency')
print('2. displacement vs mpg: DECREASING (strong negative, r ≈ -0.81)')
print('   Larger engine displacement → more fuel consumed → lower mpg')
print('3. horsepower   vs mpg: DECREASING (strong negative, r ≈ -0.78)')
print('   More horsepower → more fuel needed → lower mpg')
print('4. weight       vs mpg: DECREASING (strongest negative, r ≈ -0.83)')
print('   Heavier car → more energy needed → lower mpg')
print('5. acceleration vs mpg: INCREASING (weak positive, r ≈ +0.42)')
print('   Faster acceleration often correlates with lighter/more efficient cars')

### Step 7. Define the pipeline with pre-processing steps

Use `ColumnTransformer` to separate the preprocessing steps of the numerical features from the categorical ones.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

# Define which columns are numerical vs categorical
numerical_features   = ['displacement', 'horsepower', 'weight', 'acceleration']
categorical_features = ['cylinders', 'model year', 'origin']

# Numerical pipeline: impute missing values, then scale
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler())
])

# Categorical pipeline: impute with most frequent, then one-hot encode
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer applies each sub-pipeline to its respective columns
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline,   numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

predictor = LinearRegression()

In [ ]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", predictor),
    ]
)
print(pipe)

### Step 8. Split the dataset into train and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Total samples     : {len(df)}')
print(f'Training set size : {X_train.shape[0]} samples (80%)')
print(f'Test set size     : {X_test.shape[0]} samples (20%)')

### Step 9.a Fit the pipeline on the training set

In [ ]:
pipe.fit(X_train, y_train)
print('Pipeline fitted successfully!')

### Step 9.b Identify the learned coefficients (for each feature) and the bias term

In [ ]:
lr = pipe.named_steps['regressor']

# Reconstruct feature names after OneHotEncoding
ohe = pipe.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
all_feature_names = numerical_features + cat_feature_names

coef_df = pd.DataFrame({
    'Feature'    : all_feature_names,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('Learned Coefficients (sorted by absolute impact):')
display(coef_df)
print()
print(f'Bias (intercept): {lr.intercept_:.4f}')

### Step 9.c how much `mpg` we gain if we decrease the weight of the automobile by `100kg`?

In [ ]:
# The weight feature is scaled. We need to convert 100 units in raw space
# to scaled space: delta_scaled = delta_raw / std_weight
scaler = pipe.named_steps['preprocessor'].named_transformers_['num'].named_steps['scaler']

# numerical_features = ['displacement', 'horsepower', 'weight', 'acceleration']
weight_idx  = numerical_features.index('weight')  # index 2
weight_std  = scaler.scale_[weight_idx]
weight_coef = lr.coef_[weight_idx]   # coefficient on scaled weight

# Decreasing weight by 100 units: delta_raw = -100
delta_raw    = -100
delta_scaled = delta_raw / weight_std
mpg_gain     = weight_coef * delta_scaled

print(f'Weight coefficient (scaled space): {weight_coef:.4f}')
print(f'Weight std (from training data)  : {weight_std:.4f}')
print()
print(f'Decrease weight by 100 units -> scaled change: {delta_scaled:.4f}')
print(f'Estimated MPG gain: {mpg_gain:.4f} mpg')

### Step 10. Evaluate the pipeline on the test set

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

score  = pipe.score(X_test, y_test)
y_pred = pipe.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))

print('=== Pipeline Evaluation on Test Set ===')
print(f'  R²   : {score:.4f}  ({score*100:.1f}% variance explained)')
print(f'  MAE  : {mae:.4f} mpg')
print(f'  RMSE : {rmse:.4f} mpg')
print()
print('The ColumnTransformer pipeline correctly handles:')
print('  - Missing values in horsepower (imputed with mean)')
print('  - Scaling of numerical features (StandardScaler)')
print('  - One-hot encoding of cylinders, model year, and origin')